<a href="https://colab.research.google.com/github/G-Thor/T-715-SPPR/blob/master/ESPnetEZ/TTS_finetune_vctk_dump.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning VITS for Text-to-Speech Synthesis on a New Dataset
In this tutorial, we will guide you through the process of performing text-to-speech (TTS) synthesis by fine-tuning the VITS model on the VCTK dataset. This demo covers data preparation from dump files, model fine-tuning, inference, and evaluation.

## Overview
- Task: Text-to-Speech (TTS)
- Dataset: [VCTK](http://www.udialogue.org/download/cstr-vctk-corpus.html)
- Model: VITS - [espnet/kan-bayashi_libritts_xvector_vits](https://huggingface.co/espnet/kan-bayashi_libritts_xvector_vits)

## License Reminder
Before proceeding, please note that the dataset and model used in this tutorial come with specific licensing terms:
- **VCTK Corpus:** Licensed under the Open Data Commons Attribution License (ODC-By) v1.0.
- **Model:** The pretrained VITS model is under the Creative Commons Attribution 4.0 License.


# Prepare Environment

## Clone ESPnet's Repository

In [1]:
!git clone https://github.com/espnet/espnet.git

Cloning into 'espnet'...
remote: Enumerating objects: 739234, done.
remote: Counting objects: 100% (21933/21933), done.
remote: Compressing objects: 100% (6243/6243), done.
remote: Total 739234 (delta 21765), reused 15745 (delta 15689), pack-reused 717301 (from 3)
Receiving objects: 100% (739234/739234), 1.41 GiB | 15.71 MiB/s, done.
Resolving deltas: 100% (640531/640531), done.


## Install ESPnet and Dependencies

In [2]:
# NOTE: pip shows imcompatible errors due to preinstalled libraries but you do not need to care
# ESPnet installation
!git clone --depth 5 https://github.com/espnet/espnet.git
!cd espnet && pip install .

!pip install espnet_model_zoo tensorboard

!pip install pyopenjtalk==0.4
!pip install pypinyin==0.44.0
!pip install gdown==4.4.0
!pip install ipywebrtc

# Evaluation related
!git clone --depth 5 https://github.com/shinjiwlab/versa.git
!cd versa && pip install .
!git clone https://github.com/ftshijt/versa_demo_egs.git

import nltk
nltk.download('averaged_perceptron_tagger_eng')

fatal: destination path 'espnet' already exists and is not an empty directory.
Processing /content/espnet
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached setuptools-73.0.1-py3-none-any.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
Using cached setuptools-73.0.1-py3-none-any.whl (2.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 7.3 MB/s eta 0:00

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

## Import ESPnetEZ

In [3]:
import espnetez as ez

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


# Data Preparation

In this tutorial, we will use ESPnet-generated dump files as our inputs. Set up the directory where your processed dump folder is stored.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
DUMP_DIR = f"/content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump"
data_info = {
    "speech": ["wav.scp", "sound"],
    "text": ["text", "text"],
}

# Fine-Tuning

## Download Pretrained VITS Model
We'll use ESPnet's model zoo to download the [pretrained VITS model from the LibriTTS corpus](https://huggingface.co/espnet/kan-bayashi_libritts_xvector_vits).


In [7]:
from espnet_model_zoo.downloader import ModelDownloader

PRETRAIN_MODEL = "espnet/kan-bayashi_libritts_xvector_vits"
d = ModelDownloader()
pretrain_downloaded = d.download_and_unpack(PRETRAIN_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

## Configure Fine-Tuning

Load the pretrained model's configuration and set it up for fine-tuning.

In [13]:
TASK = "gan_tts"

pretrain_config = ez.config.from_yaml(TASK, pretrain_downloaded["train_config"])

# Update the configuration with the downloaded model file path
pretrain_config["model_file"] = pretrain_downloaded["model_file"]

# Modify configuration for fine-tuning
finetune_config = pretrain_config.copy()
finetune_config["batch_size"] = 1
finetune_config["num_workers"] = 0
finetune_config["max_epoch"] = 10
finetune_config["batch_bins"] = 500000
finetune_config["num_iters_per_epoch"] = None
finetune_config["generator_first"] = True

# Disable distributed training
finetune_config["distributed"] = False
finetune_config["multiprocessing_distributed"] = False
finetune_config["dist_world_size"] = None
finetune_config["dist_rank"] = None
finetune_config["local_rank"] = None
finetune_config["dist_master_addr"] = None
finetune_config["dist_master_port"] = None
finetune_config["dist_launcher"] = None

## Initialize Trainer

Define the trainer for the fine-tuning process.

In [14]:
DATASET_NAME = "vctk"
EXP_DIR = f"./exp/finetune_{TASK}_{DATASET_NAME}"
STATS_DIR = f"./exp/stats_{DATASET_NAME}"
ngpu = 1

trainer = ez.Trainer(
    task=TASK,
    train_config=finetune_config,
    train_dump_dir=f"{DUMP_DIR}/raw/tr_no_dev",
    valid_dump_dir=f"{DUMP_DIR}/raw/dev",
    data_info=data_info,
    output_dir=EXP_DIR,
    stats_dir=STATS_DIR,
    ngpu=ngpu,
)

# Add the xvector paths to the configuration
trainer.train_config.train_data_path_and_name_and_type += [
    [f"{DUMP_DIR}/xvector/tr_no_dev/spk_embed.scp", "spembs", "kaldi_ark"],
]
trainer.train_config.valid_data_path_and_name_and_type += [
    [f"{DUMP_DIR}/xvector/dev/spk_embed.scp", "spembs", "kaldi_ark"],
]

### Adjust `wav.scp` Paths

The `wav.scp` files generated for ESPnet might contain relative paths. Since the data is located on Google Drive, we need to ensure that the paths within these files are absolute, pointing directly to the audio files on the mounted drive. The `LibsndfileError` indicates that `soundfile` is failing to open the audio files because it's looking for them in the wrong place.

In [25]:
import os

def fix_kaldi_scp_paths(scp_file_path, base_path_for_relatives):
    """
    Reads an .scp file, prepends a base_path_for_relatives to all relative paths, and writes it back.
    """
    print(f"Fixing paths in: {scp_file_path}")
    fixed_lines = []
    try:
        with open(scp_file_path, 'r') as f:
            for line in f:
                parts = line.strip().split(maxsplit=1)
                if len(parts) == 2:
                    audio_id, path_entry = parts
                    # Split path_entry by ':' to handle Kaldi's ark:offset format
                    path_to_file = path_entry.split(':', 1)[0]
                    offset_part = ':' + path_entry.split(':', 1)[1] if ':' in path_entry else ''

                    # Check if the path is already absolute, if not, prepend the prefix
                    if not os.path.isabs(path_to_file) and not path_to_file.startswith('/content/drive'):
                        fixed_path_to_file = os.path.join(base_path_for_relatives, path_to_file)
                        fixed_lines.append(f"{audio_id} {fixed_path_to_file}{offset_part}")
                    else:
                        fixed_lines.append(line.strip())
                else:
                    fixed_lines.append(line.strip())

        with open(scp_file_path, 'w') as f:
            for line in fixed_lines:
                f.write(f"{line}\n")
        print(f"Successfully fixed paths in {scp_file_path}")
    except FileNotFoundError:
        print(f"Warning: {scp_file_path} not found. Skipping path fixing for this file.")



In [33]:
# For wav.scp files, the paths within them are relative to the 'Module 2' folder.
WAV_FILES_BASE_PATH = os.path.dirname(os.path.dirname(DUMP_DIR)) # Should be /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/
fix_kaldi_scp_paths(f"{DUMP_DIR}/raw/tr_no_dev/wav.scp", WAV_FILES_BASE_PATH)
fix_kaldi_scp_paths(f"{DUMP_DIR}/raw/dev/wav.scp", WAV_FILES_BASE_PATH)
fix_kaldi_scp_paths(f"{DUMP_DIR}/raw/eval1/wav.scp", WAV_FILES_BASE_PATH)

# For spk_embed.scp files, the paths within them are relative to the DUMP_DIR.
XVECTOR_FILES_BASE_PATH = DUMP_DIR # This should be /content/drive/MyDrive/.../gunnar/dump
fix_kaldi_scp_paths(f"{DUMP_DIR}/xvector/tr_no_dev/spk_embed.scp", XVECTOR_FILES_BASE_PATH)
fix_kaldi_scp_paths(f"{DUMP_DIR}/xvector/dev/spk_embed.scp", XVECTOR_FILES_BASE_PATH)
fix_kaldi_scp_paths(f"{DUMP_DIR}/xvector/eval1/spk_embed.scp", XVECTOR_FILES_BASE_PATH)

Fixing paths in: /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/tr_no_dev/wav.scp
Successfully fixed paths in /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/tr_no_dev/wav.scp
Fixing paths in: /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/dev/wav.scp
Successfully fixed paths in /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/dev/wav.scp
Fixing paths in: /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/eval1/wav.scp
Fixing paths in: /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/xvector/tr_no_dev/spk_embed.scp
Successfully fixed paths in /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/xvector/tr_no_dev/spk_embed.scp
Fixing paths in: /content

In [29]:
import os

# The `remove_repetitions` function has been removed as the `fix_kaldi_scp_paths`
# function now correctly ensures absolute paths, making this repetition fix redundant.
# If path issues persist, they likely stem from incorrect base paths in `fix_kaldi_scp_paths`.

In [30]:
# For wav.scp files, the paths within them are relative to the 'Module 2' folder.
remove_repetitions(f"{DUMP_DIR}/raw/tr_no_dev/wav.scp")
remove_repetitions(f"{DUMP_DIR}/raw/dev/wav.scp")
remove_repetitions(f"{DUMP_DIR}/raw/eval1/wav.scp")

# For spk_embed.scp files, the paths within them are relative to the DUMP_DIR.
remove_repetitions(f"{DUMP_DIR}/xvector/tr_no_dev/spk_embed.scp")
remove_repetitions(f"{DUMP_DIR}/xvector/dev/spk_embed.scp")
remove_repetitions(f"{DUMP_DIR}/xvector/eval1/spk_embed.scp")

Fixing paths in: /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/tr_no_dev/wav.scp
Successfully fixed paths in /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/tr_no_dev/wav.scp
Fixing paths in: /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/dev/wav.scp
Successfully fixed paths in /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/dev/wav.scp
Fixing paths in: /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/raw/eval1/wav.scp
Fixing paths in: /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/xvector/tr_no_dev/spk_embed.scp
Successfully fixed paths in /content/drive/MyDrive/Speech Processing - T-715-SPPR/2026/Class Projects/Module 2/gunnar/dump/xvector/tr_no_dev/spk_embed.scp
Fixing paths in: /content

## Collect Statistics

Before training, we need to collect data statistics (e.g., normalization stats).

In [32]:
# Temporarily set to None, as we need to collect stats first
trainer.train_config.normalize = None
trainer.train_config.pitch_normalize = None
trainer.train_config.energy_normalize = None

# Collect stats
trainer.collect_stats()

# Restore normalization configs with collected stats
trainer.train_config.write_collected_feats = False
if finetune_config["normalize"] is not None:
    trainer.train_config.normalize = finetune_config["normalize"]
    trainer.train_config.normalize_conf["stats_file"] = (
        f"{STATS_DIR}/train/feats_stats.npz"
    )
if finetune_config["pitch_normalize"] is not None:
    trainer.train_config.pitch_normalize = finetune_config["pitch_normalize"]
    trainer.train_config.pitch_normalize_conf["stats_file"] = (
        f"{STATS_DIR}/train/pitch_stats.npz"
    )
if finetune_config["energy_normalize"] is not None:
    trainer.train_config.energy_normalize = finetune_config["energy_normalize"]
    trainer.train_config.energy_normalize_conf["stats_file"] = (
        f"{STATS_DIR}/train/energy_stats.npz"
    )

/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-b92802b2-937c-4400-be38-3e8808a977cc.json


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf0 in position 0: unexpected end of data

## Start Training

Now, let's start the fine-tuning process.

In [43]:
import torch, functools
torch.load = functools.partial(torch.load, weights_only=False)

In [44]:
trainer.train()

/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-2662e5a7-6ec5-4105-b868-c17f1e49fb82.json
/usr/local/lib/python3.12/dist-packages/espnet2/train/gan_trainer.py:174: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(scaler is not None):
/usr/local/lib/python3.12/dist-packages/espnet2/gan_tts/espnet_model.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(False):
/usr/local/lib/python3.12/dist-packages/espnet2/gan_tts/vits/vits.py:429: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/usr/local/lib/python3.12/dist-packages/espnet2/gan_tts/vits/vits.py:543: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Pleas

OSError: [Errno 28] No space left on device: 'exp/finetune_gan_tts_vctk/checkpoint.pth' -> 'exp/finetune_gan_tts_vctk/checkpoint_58.pth'

# Inference

When training is done, we can use the inference API to synthesize audio from the test set.

In [ ]:
from espnet2.bin.tts_inference import inference

ckpt_name = "train.total_count.ave_10best"
inference_folder = f"{EXP_DIR}/inference_{ckpt_name}"
model_file = f"{EXP_DIR}/{ckpt_name}.pth"

inference(
    output_dir=inference_folder,
    batch_size=1,
    dtype="float32",
    ngpu=0,
    seed=0,
    num_workers=1,
    log_level="INFO",
    data_path_and_name_and_type=[
        (f"{DUMP_DIR}/raw/eval1/text", "text", "text"),
        (f"{DUMP_DIR}/raw/eval1/wav.scp", "speech", "sound"),
        (f"{DUMP_DIR}/xvector/eval1/spk_embed.scp", "spembs", "kaldi_ark"),
    ],
    key_file=None,
    train_config=f"{EXP_DIR}/config.yaml",
    model_file=model_file,
    model_tag=None,
    threshold=0.5,
    minlenratio=0.0,
    maxlenratio=10.0,
    use_teacher_forcing=False,
    use_att_constraint=False,
    backward_window=1,
    forward_window=3,
    speed_control_alpha=1.0,
    noise_scale=0.667,
    noise_scale_dur=0.8,
    always_fix_seed=False,
    allow_variable_data_keys=False,
    vocoder_config=None,
    vocoder_file=None,
    vocoder_tag=None,
)

## Inference with Pre-trained Model on Fine-tuning Eval Set

To compare the performance, we will now run inference using the original pre-trained VITS model on the same evaluation dataset used for fine-tuning.

In [ ]:
pretrain_inference_folder = f"{EXP_DIR}/inference_pretrain_eval"

inference(
    output_dir=pretrain_inference_folder,
    batch_size=1,
    dtype="float32",
    ngpu=0,
    seed=0,
    num_workers=1,
    log_level="INFO",
    data_path_and_name_and_type=[
        (f"{DUMP_DIR}/raw/eval1/text", "text", "text"),
        (f"{DUMP_DIR}/raw/eval1/wav.scp", "speech", "sound"),
        (f"{DUMP_DIR}/xvector/eval1/spk_embed.scp", "spembs", "kaldi_ark"),
    ],
    key_file=None,
    train_config=pretrain_downloaded["train_config"],
    model_file=pretrain_downloaded["model_file"],
    model_tag=None,
    threshold=0.5,
    minlenratio=0.0,
    maxlenratio=10.0,
    use_teacher_forcing=False,
    use_att_constraint=False,
    backward_window=1,
    forward_window=3,
    speed_control_alpha=1.0,
    noise_scale=0.667,
    noise_scale_dur=0.8,
    always_fix_seed=False,
    allow_variable_data_keys=False,
    vocoder_config=None,
    vocoder_file=None,
    vocoder_tag=None,
)

## Evaluation of Pre-trained Model

Now, let's evaluate the audio synthesized by the pre-trained model on the fine-tuning evaluation set.

In [ ]:
import soundfile as sf
from versa import speaker_metric, speaker_model_setup, mcd_f0

gt_wav_scp = f"{DUMP_DIR}/raw/eval1/wav.scp"

# Ensure the speaker model is set up if not already in the global scope
# model = speaker_model_setup()

spk_similarities_pretrain = []
mcd_f0s_pretrain = []
f0rmses_pretrain = []
f0corrs_pretrain = []

with open(gt_wav_scp, "r") as f:
    for line in f:
        key, path = line.strip().split()
        gt, sr = sf.read(path)
        # Use the pretrain_inference_folder for predictions
        pred, sr = sf.read(f"{pretrain_inference_folder}/wav/{key}.wav")
        ret = speaker_metric(model, pred, gt, sr)
        with open(f"{pretrain_inference_folder}/spk_similarity", "a") as f:
            f.write(f"{ret['spk_similarity']}\n")
        spk_similarities_pretrain.append(ret["spk_similarity"])
        ret = mcd_f0(pred, gt, sr, 1, 800, dtw=True)
        with open(f"{pretrain_inference_folder}/mcd_f0", "a") as f:
            f.write(f"{ret['mcd']}\n")
        with open(f"{pretrain_inference_folder}/f0rmse", "a") as f:
            f.write(f"{ret['f0rmse']}\n")
        with open(f"{pretrain_inference_folder}/f0corr", "a") as f:
            f.write(f"{ret['f0corr']}\n")
        mcd_f0s_pretrain.append(ret["mcd"])
        f0rmses_pretrain.append(ret["f0rmse"])
        f0corrs_pretrain.append(ret["f0corr"])

print("--- Pre-trained Model Evaluation on Fine-tuning Eval Set ---")
print("Averaged speaker similarity:", sum(spk_similarities_pretrain) / len(spk_similarities_pretrain))
print("Averaged MCD:", sum(mcd_f0s_pretrain) / len(mcd_f0s_pretrain))
print("Averaged F0 RMSE:", sum(f0rmses_pretrain) / len(f0rmses_pretrain))
print("Averaged F0 Corr:", sum(f0corrs_pretrain) / len(f0corrs_pretrain))

# Evaluation
In this section, we will assess the model's performance based on speaker similarity, Mel-cepstral distortion, the root mean square error (RMSE) of the fundamental frequency (f0), and the Pearson correlation coefficient for f0.

In [ ]:
import soundfile as sf
from versa import speaker_metric, speaker_model_setup, mcd_f0

gt_wav_scp = f"{DUMP_DIR}/raw/eval1/wav.scp"

model = speaker_model_setup()
spk_similarities = []
mcd_f0s = []
f0rmses = []
f0corrs = []

with open(gt_wav_scp, "r") as f:
    for line in f:
        key, path = line.strip().split()
        gt, sr = sf.read(path)
        pred, sr = sf.read(f"{inference_folder}/wav/{key}.wav")
        ret = speaker_metric(model, pred, gt, sr)
        with open(f"{inference_folder}/spk_similarity", "a") as f:
            f.write(f"{ret['spk_similarity']}\n")
        spk_similarities.append(ret["spk_similarity"])
        ret = mcd_f0(pred, gt, sr, 1, 800, dtw=True)
        with open(f"{inference_folder}/mcd_f0", "a") as f:
            f.write(f"{ret['mcd']}\n")
        with open(f"{inference_folder}/f0rmse", "a") as f:
            f.write(f"{ret['f0rmse']}\n")
        with open(f"{inference_folder}/f0corr", "a") as f:
            f.write(f"{ret['f0corr']}\n")
        mcd_f0s.append(ret["mcd"])
        f0rmses.append(ret["f0rmse"])
        f0corrs.append(ret["f0corr"])

print("Averaged speaker similarity:", sum(spk_similarities) / len(spk_similarities))
print("Averaged MCD:", sum(mcd_f0s) / len(mcd_f0s))
print("Averaged F0 RMSE:", sum(f0rmses) / len(f0rmses))
print("Averaged F0 Corr:", sum(f0corrs) / len(f0corrs))

# References

[1] S. Someki, K. Choi, S. Arora, W. Chen, S. Cornell, J. Han, Y. Peng, J. Shi, V. Srivastav, and S. Watanabe, “ESPnet-EZ: Python-only ESPnet for Easy Fine-tuning and Integration,” *arXiv preprint* arXiv:2409.09506, 2024.

[2] C. Veaux, J. Yamagishi, and K. MacDonald, “CSTR VCTK Corpus: English Multi-speaker Corpus for CSTR Voice Cloning Toolkit,” University of Edinburgh, The Centre for Speech Technology Research (CSTR), 2017. [Sound]. https://doi.org/10.7488/ds/1994.

[3] J. Shi, H. Shim, J. Tian, S. Arora, H. Wu, D. Petermann, J. Q. Yip, Y. Zhang, Y. Tang, W. Zhang, D. S. Alharthi, Y. Huang, K. Saito, J. Han, Y. Zhao, C. Donahue, and S. Watanabe, “VERSA: A Versatile Evaluation Toolkit for Speech, Audio, and Music,” arXiv preprint arXiv:2412.17667, 2024.